# Fase 0 — Pilot Crawler

**Tujuan:** memverifikasi struktur HTML DetikFinance (indeks + artikel), mengukur
volume berita & kebutuhan disk, dan menguji selektor di tiga era HTML (2022 / 2024 / 2026)
**sebelum** meng-commit ke crawl rentang penuh.

Mengacu pada spec §2.5 (aturan keputusan pilot) dan §5.1 (etika).

## Yang sudah terverifikasi (hasil pilot awal)

| Hal | Temuan |
|-----|--------|
| URL indeks | `https://finance.detik.com/indeks?page={n}&date=MM/DD/YYYY` — **server-side**, tak perlu JS |
| URL artikel | `https://finance.detik.com/{subchannel}/d-{id}/{slug}` |
| Item/halaman | ~20 artikel; hari normal ~6 halaman (~110 artikel/hari) |
| Akhir paginasi | halaman kosong → berhenti |
| Waktu terbit | `<meta name="publishdate">` (format `YYYY/MM/DD HH:MM:SS`, presisi detik) |
| Judul / body | `h1.detail__title` / `.detail__body-text` — **stabil di 2022, 2024, 2026** |
| Disk (gz) | ~48 KB/artikel, ~25 KB/halaman indeks → proyeksi skenario C ≈ **1.6 GB** |
| robots.txt | `User-agent: *` → `Allow: /`; aturan `*?date=` & `/*indeks/` hanya untuk **Googlebot** |


In [ ]:
import sys, yaml
from datetime import date, timedelta
from pathlib import Path

sys.path.insert(0, "..")
from src.crawl import (
    Crawler, CrawlConfig, save_html_gz, load_html_gz,
    index_html_path, article_html_path,
)
from src.parse import parse_index, parse_article

cfg = yaml.safe_load(open("../config/config.yaml"))
sel = cfg["selectors"]
RAW = Path("../data/raw")
cfg["pilot"], cfg["crawl"]["base_url"]

## 1. Siapkan crawler yang sopan

Untuk pilot kita percepat sedikit. **Untuk crawl produksi, gunakan `request_delay_sec=1.0`.**

In [ ]:
cc = CrawlConfig(
    user_agent=cfg["crawl"]["user_agent"],  # GANTI kontak email di config.yaml
    request_delay_sec=0.3,                   # pilot; produksi = 1.0
    jitter_sec=(0.1, 0.3),
)
cw = Crawler(cfg=cc)

## 2. Crawl beberapa hari dari tiap era + ukur disk & parse

In [ ]:
sample_days = [date(2022, 3, 1), date(2024, 3, 1), date(2026, 3, 2)]

idx_bytes = art_bytes = n_art = 0
rows = []
for d in sample_days:
    html = cw.fetch_index_page(d, page=1)
    idx_bytes += save_html_gz(html, index_html_path(RAW, d, 1))
    items = parse_index(html, sel)
    print(f"{d}: {len(items)} items on page 1")
    for it in items[:3]:
        ahtml = cw.fetch_article(it.url)
        art_bytes += save_html_gz(ahtml, article_html_path(RAW, it.article_id))
        art = parse_article(ahtml, it.url, sel)
        n_art += 1
        rows.append((d, it.subchannel, art.published_at, art.n_words, art.title[:50]))

import pandas as pd
pd.DataFrame(rows, columns=["index_date", "subchannel", "published_at", "n_words", "title"])

In [ ]:
print(f"index   : ~{idx_bytes/3/1024:.1f} KB/page (gz)")
print(f"article : ~{art_bytes/n_art/1024:.1f} KB/article (gz)")
proj = (idx_bytes/3*17000 + art_bytes/n_art*25000)/(1024**3)
print(f"PROYEKSI skenario C (gz) : ~{proj:.2f} GB total")

## 3. Kedalaman paginasi untuk satu hari penuh

Jalankan untuk mengukur kepadatan berita harian (bahan proyeksi volume §2.5).

In [ ]:
def crawl_full_day(day):
    all_items = []
    for page in range(1, 50):
        html = cw.fetch_index_page(day, page)
        items = parse_index(html, sel)
        if not items:
            break
        all_items.extend(items)
    return all_items

items = crawl_full_day(date(2024, 3, 1))
print(f"Total artikel 2024-03-01: {len(items)}")
pd.Series([i.subchannel for i in items]).value_counts().head(12)

## 4. Tabel keputusan pilot (spec §2.5)

Isi kolom **Hasil** setelah menjalankan pilot 3 bulan penuh, lalu putuskan:

| Metrik | Ambang | Hasil | Tindakan jika gagal |
|--------|--------|-------|---------------------|
| Arsip tercapai sampai tanggal terawal | ada | ? | turun ke opsi B/A |
| Artikel Tier A+B / bank / hari bursa | ≥ 1.0 | ? | perpanjang rentang / tambah sumber |
| % bank-hari dengan ≥1 artikel | ≥ 60% | ? | idem |
| Gagal parse (judul/tanggal/isi) | ≤ 2% | ? | perbaiki selektor per era |
| Presisi Tier A (cek 100 manual) | ≥ 90% | ? | perketat aturan alias |
| Rasio duplikat | dicatat | ? | set parameter dedup |

## ⚠️ Keputusan etika yang perlu kamu konfirmasi

`robots.txt` DetikFinance mem-`Disallow` `*?date=` dan `/*indeks/` **khusus untuk Googlebot**,
sedangkan `User-agent: *` di-`Allow: /`. Secara teknis crawler riset generik diizinkan,
tetapi praktik etis yang aman: (a) tetap 1 req/detik + jitter, (b) identifikasi riset di UA,
(c) hanya simpan untuk riset & tidak republikasi teks penuh. **Konfirmasikan ke dosen** sebelum
crawl penuh.